# AI_EMBED + Cortex Search — Semantic Search and RAG

## Use Case Overview

This example demonstrates two complementary approaches to building AI-powered search in Snowflake:

1. **Manual vector search with `AI_EMBED`** — generate embeddings yourself and use `VECTOR_COSINE_SIMILARITY` to find similar documents. Full control, no managed service.

2. **Cortex Search Service** — a fully managed semantic search service. Just point it at a table and query it. No embedding management, no index tuning.

3. **RAG (Retrieval-Augmented Generation)** — combine search retrieval with `CORTEX.COMPLETE` to answer questions grounded in your data.

**SA use case:** Enable employees, customers, or AI agents to search company knowledge bases, documentation, and internal content using natural language — with LLM-generated answers rather than just ranked links.

**Dataset:** The 5 `wiki_articles` from the `ai-summarize-articles` example (reused from Phase 3a).

**Prerequisites:** Run `unstructured/ai-summarize-articles/setup.sql` first.

In [ ]:
import os
import json
import requests
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Part 1 — Manual Vector Search with AI_EMBED

`SNOWFLAKE.CORTEX.EMBED_TEXT_1024(model, text)` converts text into a 1024-dimension vector.
`VECTOR_COSINE_SIMILARITY(v1, v2)` measures how semantically similar two vectors are (0 to 1).

This approach gives you full control: store embeddings in your own table, update them on your schedule, query them with any SQL logic.

In [ ]:
embeddings_df = pd.read_sql("SELECT article_id, title, category FROM article_embeddings ORDER BY article_id", conn)
print("Embeddings table:")
embeddings_df

### Semantic Search with Vector Similarity

Embed the user query and find the most similar articles by cosine similarity.

In [ ]:
user_query = "How do AI assistants store and retrieve knowledge?"

semantic_results = pd.read_sql(f"""
    SELECT
        title,
        category,
        VECTOR_COSINE_SIMILARITY(
            embedding,
            SNOWFLAKE.CORTEX.EMBED_TEXT_1024('e5-base-v2', '{user_query}')
        ) AS similarity_score
    FROM article_embeddings
    ORDER BY similarity_score DESC
    LIMIT 3
""", conn)
print(f"Query: '{user_query}'\n")
semantic_results

## Part 2 — Cortex Search Service

Cortex Search is a managed service — you create it once and query it via the REST API or Python SDK. It handles indexing, embedding updates, and approximate nearest-neighbor search automatically.

This is the recommended approach for production use cases.

In [ ]:
from snowflake.core import Root

root = Root(conn)
search_service = (
    root.databases["GENAI_LEARNING"]
        .schemas["PUBLIC"]
        .cortex_search_services["WIKI_SEARCH"]
)

def search_articles(query: str, limit: int = 3) -> list[dict]:
    response = search_service.search(
        query=query,
        columns=["title", "category", "body"],
        limit=limit,
    )
    return response.results

results = search_articles("machine learning model training and deployment")
for r in results:
    print(f"\nTitle: {r['title']} ({r['category']})")
    print(f"Excerpt: {r['body'][:200]}...")

## Part 3 — RAG: Search → Retrieve → Generate

RAG (Retrieval-Augmented Generation) combines search with LLM generation:
1. **Retrieve**: search for the most relevant documents to a user question
2. **Augment**: inject the retrieved documents into the LLM prompt as context
3. **Generate**: ask the LLM to answer the question using only the provided context

This grounds LLM responses in your actual data, reducing hallucinations and enabling knowledge-base Q&A.

In [ ]:
def rag_answer(question: str, top_k: int = 2) -> str:
    retrieved = search_articles(question, limit=top_k)
    context = "\n\n---\n\n".join(
        f"Source: {r['title']}\n{r['body']}" for r in retrieved
    )

    prompt = f"""You are a helpful assistant. Answer the question below using ONLY the provided context documents.
If the answer is not in the context, say "I don't have enough information to answer that."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    result = pd.read_sql(
        f"SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-haiku', $${prompt}$$) AS answer",
        conn
    )
    return result["ANSWER"][0]


questions = [
    "What is the difference between a data warehouse and a data lakehouse?",
    "How does RAG work and why is it useful?",
    "What was the Snowflake IPO and when did it happen?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")

## Summary — Manual Embeddings vs. Cortex Search Service

| Aspect | Manual (AI_EMBED) | Cortex Search Service |
|---|---|---|
| Setup | Create + manage embeddings table | `CREATE CORTEX SEARCH SERVICE` |
| Updates | Manual re-embedding on change | Automatic with `TARGET_LAG` |
| Query interface | SQL (`VECTOR_COSINE_SIMILARITY`) | Python SDK or REST API |
| Control | Full (custom chunking, filtering) | Managed |
| Best for | Custom pipelines, one-off analysis | Production search applications |

**SA tips:**
- Use Cortex Search for **Cortex Agent tools** — agents can query search services as a tool
- For large document sets, chunk documents into smaller pieces before embedding (each chunk ≈ 200–500 tokens)
- Store both the chunk and the parent document reference so you can retrieve the full article after a chunk matches
- Add metadata columns (`category`, `source_url`, `last_updated`) to the Search Service for filtered search